# Notebook 11 – Machine Learning Model Comparison for Dengue Case Forecasting

## Objective

The objective of this notebook is to compare different machine learning algorithms for forecasting dengue case counts in the next reporting period across 25 districts in Sri Lanka.

The comparison will use the same forecasting target, baseline predictors, and chronological training, validation, and testing partitions to ensure a fair evaluation.

## Algorithms

1. Linear Regression
2. Random Forest Regressor
3. HistGradientBoostingRegressor
4. XGBoost Regressor

## Evaluation Metrics

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R-squared (R²)

## Experimental Approach

Models will be trained using historical training data, compared using validation data, and the selected model will be evaluated using the untouched test dataset.

The existing HistGradientBoostingRegressor model from Notebook 06 will serve as the benchmark.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
import json
import time

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dengue_weather_weekly_25_districts.csv"
)

# Load dataset
df = pd.read_csv(DATA_PATH)

# Convert date columns
df["week_start"] = pd.to_datetime(df["week_start"])
df["week_end"] = pd.to_datetime(df["week_end"])

# Sort chronologically by district
df = df.sort_values(
    ["district_key", "week_start"]
).reset_index(drop=True)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("Number of districts:", df["district_key"].nunique())
print("Start date:", df["week_start"].min())
print("End date:", df["week_start"].max())

df.head()

Dataset loaded successfully.
Dataset shape: (24775, 27)
Number of districts: 25
Start date: 2006-12-23 00:00:00
End date: 2025-12-13 00:00:00


,district_key,week_start,week_end,dengue_cases,reporting_period_days,source_reporting_rows,source_areas,weather_days_observed,weather_days_expected,weather_days_complete,...,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,relative_humidity_2m_max,relative_humidity_2m_min,dew_point_2m_mean,wind_speed_10m_mean,wind_speed_10m_max,cloud_cover_mean,wind_direction_10m_dominant
0,ampara,2006-12-23,2006-12-29,0,7,2,Ampara | Kalmune,7,7,True,...,27.528571,23.571429,79.857143,87.285714,70.571429,21.471429,15.685714,21.085714,89.142857,8.574025
1,ampara,2006-12-30,2007-01-05,0,7,2,Ampara | Kalmune,7,7,True,...,28.042857,21.385714,71.142857,82.428571,57.000000,18.885714,14.485714,19.614286,59.428571,358.285579
2,ampara,2007-01-06,2007-01-12,0,7,2,Ampara | Kalmune,7,7,True,...,26.957143,23.271429,86.285714,92.285714,78.714286,22.285714,14.842857,18.828571,78.285714,20.694632
3,ampara,2007-01-13,2007-01-19,0,7,2,Ampara | Kalmune,7,7,True,...,27.728571,23.571429,83.142857,89.714286,74.571429,22.357143,17.114286,23.000000,77.857143,11.275426
4,ampara,2007-01-20,2007-01-26,0,7,2,Ampara | Kalmune,7,7,True,...,27.814286,22.742857,76.714286,85.142857,66.142857,20.600000,15.514286,20.385714,71.142857,7.286656


In [3]:
# Create a copy for forecasting feature engineering

forecast_df = df.copy()

# 1. Current-period dengue cases
forecast_df["dengue_current_period"] = (
    forecast_df["dengue_cases"]
)

# 2. Historical dengue lag features
lag_periods = [1, 2, 3, 4, 51, 52]

for lag in lag_periods:
    forecast_df[f"dengue_lag_{lag}"] = (
        forecast_df.groupby("district_key")["dengue_cases"]
        .shift(lag)
    )

# 3. Four-period rolling mean
forecast_df["dengue_rolling_mean_4"] = (
    forecast_df.groupby("district_key")["dengue_cases"]
    .transform(
        lambda s: s.rolling(
            window=4,
            min_periods=2
        ).mean()
    )
)

# 4. Calendar features
week_of_year = (
    forecast_df["week_start"]
    .dt.isocalendar()
    .week.astype(int)
)

forecast_df["month"] = forecast_df["week_start"].dt.month

forecast_df["week_sin"] = np.sin(
    2 * np.pi * week_of_year / 52.1775
)

forecast_df["week_cos"] = np.cos(
    2 * np.pi * week_of_year / 52.1775
)

print("Forecasting features created successfully.")
print("Dataset shape:", forecast_df.shape)

Forecasting features created successfully.
Dataset shape: (24775, 38)


In [4]:
base_features = [
    "district_key",
    "dengue_current_period",
    "dengue_lag_1",
    "dengue_lag_2",
    "dengue_lag_3",
    "dengue_lag_4",
    "dengue_lag_51",
    "dengue_lag_52",
    "dengue_rolling_mean_4",
    "month",
    "week_sin",
    "week_cos"
]

print("Number of predictors:", len(base_features))
print(base_features)

Number of predictors: 12
['district_key', 'dengue_current_period', 'dengue_lag_1', 'dengue_lag_2', 'dengue_lag_3', 'dengue_lag_4', 'dengue_lag_51', 'dengue_lag_52', 'dengue_rolling_mean_4', 'month', 'week_sin', 'week_cos']


In [6]:
print("Target columns available:")

print("target_next_period:",
      "target_next_period" in forecast_df.columns)

print("target_week_start:",
      "target_week_start" in forecast_df.columns)

print("\nLast available columns:")
print(forecast_df.columns.tolist()[-10:])

Target columns available:
target_next_period: False
target_week_start: False

Last available columns:
['dengue_lag_1', 'dengue_lag_2', 'dengue_lag_3', 'dengue_lag_4', 'dengue_lag_51', 'dengue_lag_52', 'dengue_rolling_mean_4', 'month', 'week_sin', 'week_cos']


In [7]:
# Sort data chronologically within each district
forecast_df = forecast_df.sort_values(
    ["district_key", "week_start"]
).reset_index(drop=True)

# Create target columns
forecast_df["target_next_period"] = (
    forecast_df.groupby("district_key")["dengue_cases"].shift(-1)
)

forecast_df["target_week_start"] = (
    forecast_df.groupby("district_key")["week_start"].shift(-1)
)

# Verify
print("Target columns created:")
print(forecast_df[
    ["district_key", "week_start",
     "target_next_period", "target_week_start"]
].head())

print("\nTarget columns exist:",
      "target_next_period" in forecast_df.columns and
      "target_week_start" in forecast_df.columns)

Target columns created:
  district_key week_start  target_next_period target_week_start
0       ampara 2006-12-23                 0.0        2006-12-30
1       ampara 2006-12-30                 0.0        2007-01-06
2       ampara 2007-01-06                 0.0        2007-01-13
3       ampara 2007-01-13                 0.0        2007-01-20
4       ampara 2007-01-20                 0.0        2007-01-27

Target columns exist: True


In [8]:
forecast_final = forecast_df.dropna(
    subset=["target_next_period", "target_week_start"]
).copy()

forecast_final = forecast_final.sort_values(
    ["target_week_start", "district_key"]
).reset_index(drop=True)

print("Final dataset shape:", forecast_final.shape)
print("Number of districts:", forecast_final["district_key"].nunique())
print("Missing targets:", forecast_final["target_next_period"].isna().sum())
print("Duplicate district-target dates:",
      forecast_final.duplicated(
          ["district_key", "target_week_start"]
      ).sum())

Final dataset shape: (24750, 40)
Number of districts: 25
Missing targets: 0
Duplicate district-target dates: 0


In [10]:
# Chronological split using target reporting dates

train_mask = (
    forecast_final["target_week_start"]
    <= pd.Timestamp("2020-04-04")
)

valid_mask = (
    (forecast_final["target_week_start"] >= pd.Timestamp("2020-04-11")) &
    (forecast_final["target_week_start"] <= pd.Timestamp("2023-02-04"))
)

test_mask = (
    (forecast_final["target_week_start"] >= pd.Timestamp("2023-02-11")) &
    (forecast_final["target_week_start"] <= pd.Timestamp("2025-12-13"))
)

train = forecast_final.loc[train_mask].copy()
valid = forecast_final.loc[valid_mask].copy()
test = forecast_final.loc[test_mask].copy()

print("Training rows:", len(train))
print("Validation rows:", len(valid))
print("Testing rows:", len(test))

print("\nTraining date range:")
print(train["target_week_start"].min(), "to",
      train["target_week_start"].max())

print("\nValidation date range:")
print(valid["target_week_start"].min(), "to",
      valid["target_week_start"].max())

print("\nTesting date range:")
print(test["target_week_start"].min(), "to",
      test["target_week_start"].max())

Training rows: 17325
Validation rows: 3700
Testing rows: 3725

Training date range:
2006-12-30 00:00:00 to 2020-04-04 00:00:00

Validation date range:
2020-04-11 00:00:00 to 2023-02-04 00:00:00

Testing date range:
2023-02-11 00:00:00 to 2025-12-13 00:00:00


In [11]:
# Define the 12 baseline predictors

base_features = [
    "district_key",
    "dengue_current_period",
    "dengue_lag_1",
    "dengue_lag_2",
    "dengue_lag_3",
    "dengue_lag_4",
    "dengue_lag_51",
    "dengue_lag_52",
    "dengue_rolling_mean_4",
    "month",
    "week_sin",
    "week_cos"
]

target = "target_next_period"

# Separate predictors and targets

X_train = train[base_features].copy()
y_train = train[target].copy()

X_valid = valid[base_features].copy()
y_valid = valid[target].copy()

X_test = test[base_features].copy()
y_test = test[target].copy()

print("Training predictors:", X_train.shape)
print("Validation predictors:", X_valid.shape)
print("Testing predictors:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Validation target:", y_valid.shape)
print("Testing target:", y_test.shape)

Training predictors: (17325, 12)
Validation predictors: (3700, 12)
Testing predictors: (3725, 12)

Training target: (17325,)
Validation target: (3700,)
Testing target: (3725,)


In [12]:
# Define categorical and numerical features

categorical_features = ["district_key"]

numeric_features = [
    feature for feature in base_features
    if feature not in categorical_features
]

# Numerical preprocessing

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully.
Numerical features: 11
Categorical features: 1


In [13]:
# Create Linear Regression pipeline

linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

print("Linear Regression pipeline created.")

Linear Regression pipeline created.


In [14]:
import time

start_time = time.time()

linear_model.fit(X_train, y_train)

linear_training_time = time.time() - start_time

print("Linear Regression trained successfully.")
print("Training time:", round(linear_training_time, 2), "seconds")

Linear Regression trained successfully.
Training time: 0.27 seconds


In [15]:
linear_valid_predictions = linear_model.predict(X_valid)

print("Validation predictions generated.")

print("First 10 actual values:")
print(y_valid.head(10).values)

print("\nFirst 10 predicted values:")
print(np.round(linear_valid_predictions[:10], 2))

Validation predictions generated.
First 10 actual values:
[ 7.  0.  1. 23.  8.  1.  2.  4.  4.  8.]

First 10 predicted values:
[11.17  2.77  2.83 22.36 17.03  3.34 15.11  1.87 12.19 13.9 ]


In [16]:
linear_mae = mean_absolute_error(
    y_valid,
    linear_valid_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(y_valid, linear_valid_predictions)
)

linear_r2 = r2_score(
    y_valid,
    linear_valid_predictions
)

print("LINEAR REGRESSION - VALIDATION RESULTS")
print("--------------------------------------")
print("MAE:", round(linear_mae, 4))
print("RMSE:", round(linear_rmse, 4))
print("R2:", round(linear_r2, 4))

LINEAR REGRESSION - VALIDATION RESULTS
--------------------------------------
MAE: 9.0003
RMSE: 18.9857
R2: 0.6247


In [17]:
# Store Linear Regression results

model_results = []

model_results.append({
    "Model": "Linear Regression",
    "Validation_MAE": linear_mae,
    "Validation_RMSE": linear_rmse,
    "Validation_R2": linear_r2,
    "Training_Time_Seconds": linear_training_time
})

print(pd.DataFrame(model_results))

               Model  Validation_MAE  Validation_RMSE  Validation_R2  \
0  Linear Regression        9.000268        18.985684       0.624675   

   Training_Time_Seconds  
0               0.270468  


In [18]:
random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [19]:
start_time = time.time()

random_forest_model.fit(X_train, y_train)

rf_training_time = time.time() - start_time

print("Random Forest trained successfully.")
print("Training time:", round(rf_training_time, 2), "seconds")

Random Forest trained successfully.
Training time: 7.36 seconds


In [20]:
rf_valid_predictions = random_forest_model.predict(X_valid)

rf_mae = mean_absolute_error(
    y_valid, rf_valid_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(y_valid, rf_valid_predictions)
)

rf_r2 = r2_score(
    y_valid, rf_valid_predictions
)

print("RANDOM FOREST - VALIDATION RESULTS")
print("----------------------------------")
print("MAE:", round(rf_mae, 4))
print("RMSE:", round(rf_rmse, 4))
print("R2:", round(rf_r2, 4))

RANDOM FOREST - VALIDATION RESULTS
----------------------------------
MAE: 9.16
RMSE: 20.1172
R2: 0.5786


In [21]:
model_results.append({
    "Model": "Random Forest",
    "Validation_MAE": rf_mae,
    "Validation_RMSE": rf_rmse,
    "Validation_R2": rf_r2,
    "Training_Time_Seconds": rf_training_time
})

print(
    pd.DataFrame(model_results)
    .sort_values("Validation_RMSE")
)

               Model  Validation_MAE  Validation_RMSE  Validation_R2  \
0  Linear Regression        9.000268        18.985684       0.624675   
1      Random Forest        9.159959        20.117233       0.578603   

   Training_Time_Seconds  
0               0.270468  
1               7.356237  
